# Проект M1: риск сердечного приступа — исследование, предобработка, обучение

## 1) Общая идея
У нас есть данные пациентов, и нужно предсказать **высокий/низкий риск поражения сердца**.

В датасете есть признаки разных типов:
- **Антропометрия**: возраст, BMI и т.п.
- **Привычки/образ жизни**: курение, алкоголь, сон, физ.активность.
- **Давление**: систолическое/диастолическое.
- **Хронические заболевания**: например, диабет.
- **Биохимия крови**: холестерин, триглицериды и т.п.

Целевая переменная в train: `Heart Attack Risk (Binary)` (0/1).

## 2) Важное про утечку (leakage)
В медицинских данных часто бывает косвенная утечка:
- Некоторые анализы/показатели могут быть измерены после наступления события или уже при подозрении на него.
- Тогда модель "угадывает" не риск заранее, а следствие.


Ниже мы сделаем простые проверки:
- пропуски/типы
- бесполезные признаки (константные)
- сильные корреляции
- признаки, которые слишком хорошо "объясняют" таргет (как кандидаты на утечку)


In [1]:
import pandas as pd
import numpy as np

from pathlib import Path

from catboost import CatBoostClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, roc_auc_score, classification_report

_cwd = Path.cwd()
project_root = _cwd.parent if _cwd.name == "notebooks" else _cwd
train_path = project_root / "data" / "heart_train.csv"
test_path = project_root / "data" / "heart_test.csv"

train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)

target_col = "Heart Attack Risk (Binary)"
id_col = "id"

print("train:", train_df.shape)
print("test:", test_df.shape)
print("target in train:", target_col in train_df.columns)
print("target in test:", target_col in test_df.columns)

train: (8685, 28)
test: (966, 27)
target in train: True
target in test: False


## 3) Первичный осмотр данных (EDA)

Цель EDA здесь простая:
- понять, какие колонки есть
- сколько пропусков
- какой баланс классов у таргета
- нет ли явных проблем (например, странные типы в `Gender`)


In [2]:
print("Columns:")
print(list(train_df.columns))

print("\nTarget distribution:")
print(train_df[target_col].value_counts(dropna=False))

print("\nMissing values (top 15):")
print(train_df.isna().sum().sort_values(ascending=False).head(15))

print("\nDtypes (top 15):")
print(train_df.dtypes.head(15))

if "Gender" in train_df.columns:
    print("\nGender unique (sample):")
    print(sorted(train_df["Gender"].dropna().unique().tolist())[:20])

Columns:
['Unnamed: 0', 'Age', 'Cholesterol', 'Heart rate', 'Diabetes', 'Family History', 'Smoking', 'Obesity', 'Alcohol Consumption', 'Exercise Hours Per Week', 'Diet', 'Previous Heart Problems', 'Medication Use', 'Stress Level', 'Sedentary Hours Per Day', 'Income', 'BMI', 'Triglycerides', 'Physical Activity Days Per Week', 'Sleep Hours Per Day', 'Heart Attack Risk (Binary)', 'Blood sugar', 'CK-MB', 'Troponin', 'Gender', 'Systolic blood pressure', 'Diastolic blood pressure', 'id']

Target distribution:
Heart Attack Risk (Binary)
0.0    5672
1.0    3013
Name: count, dtype: int64

Missing values (top 15):
Stress Level                       243
Diabetes                           243
Family History                     243
Smoking                            243
Obesity                            243
Alcohol Consumption                243
Physical Activity Days Per Week    243
Previous Heart Problems            243
Medication Use                     243
Unnamed: 0                           

## 4) Предобработка 

Что сделаем:
1) Удалим `Unnamed: 0`, если есть (это служебный индекс).
2) `Gender` приведём к строке (в данных встречаются и `Male/Female`, и коды вроде `0.0/1.0`).
3) Остальные столбцы оставим как есть — CatBoost умеет NaN.


In [3]:
def preprocess(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    # убираем служебную колонку
    if "Unnamed: 0" in df.columns:
        df = df.drop(columns=["Unnamed: 0"])
    # приводим пол к строке
    if "Gender" in df.columns:
        df["Gender"] = df["Gender"].fillna("NA").astype(str)
        df["Gender"] = df["Gender"].replace({"nan": "NA"})
    return df

train_p = preprocess(train_df)
test_p = preprocess(test_df)

print("After preprocess:")
print("train:", train_p.shape)
print("test:", test_p.shape)

After preprocess:
train: (8685, 27)
test: (966, 26)


## 5) Бесполезные признаки

Примеры "бесполезных" признаков:
- константные (одно значение почти у всех)
- полностью пустые

In [4]:
feature_cols_all = [c for c in train_p.columns if c not in [target_col, id_col]]

nunique = train_p[feature_cols_all].nunique(dropna=False).sort_values()
print("Small nunique (<= 2) features:")
print(nunique[nunique <= 2])

Small nunique (<= 2) features:
Series([], dtype: int64)


## 6) Сильные корреляции

Сильно коррелированные признаки могут:
- ухудшать интерпретацию
- дублировать информацию

Для деревьев/бустинга это обычно не критично, но полезно знать.

Сделаем простую проверку по числовым колонкам: найдём пары с корреляцией > 0.95.

In [5]:
numeric_cols = [c for c in feature_cols_all if c != "Gender"]

corr = train_p[numeric_cols + [target_col]].corr(numeric_only=True)
corr_abs = corr.abs()

pairs = []
cols = list(corr_abs.columns)
for i in range(len(cols)):
    for j in range(i + 1, len(cols)):
        a, b = cols[i], cols[j]
        if a == target_col or b == target_col:
            continue
        if corr_abs.loc[a, b] > 0.95:
            pairs.append((a, b, float(corr.loc[a, b])))

pairs_sorted = sorted(pairs, key=lambda x: abs(x[2]), reverse=True)
print("Highly correlated pairs (|corr| > 0.95):")
for a, b, c in pairs_sorted[:30]:
    print(f"{a}  <->  {b}: corr={c:.3f}")

Highly correlated pairs (|corr| > 0.95):


## 7) Подозрение на утечку таргета

Полностью доказать утечку без описания процесса сбора данных нельзя.
Но можно искать **подозрительные признаки**:
- которые являются *следствием* события
- или которые почти идеально разделяют классы

В этом датасете особенно подозрительными выглядят:
- `Troponin`
- `CK-MB`

Проверим простую вещь: корреляция с таргетом (по числовым колонкам).

In [6]:
target_corr = corr[target_col].drop(labels=[target_col]).sort_values(key=lambda s: s.abs(), ascending=False)
print("Top correlations with target (by abs):")
print(target_corr.head(15))

leak_candidates = [c for c in ["Troponin", "CK-MB"] if c in train_p.columns]
print("\nLeak candidates present:", leak_candidates)

Top correlations with target (by abs):
Diet                              -0.044135
Systolic blood pressure            0.033762
Sleep Hours Per Day               -0.019487
Cholesterol                        0.018920
Diabetes                           0.016213
Heart rate                        -0.015561
Obesity                           -0.015084
Alcohol Consumption               -0.014546
Physical Activity Days Per Week   -0.012515
Triglycerides                      0.012062
Diastolic blood pressure           0.011967
CK-MB                             -0.009821
Exercise Hours Per Week            0.009487
Income                             0.008218
Troponin                           0.007608
Name: Heart Attack Risk (Binary), dtype: float64

Leak candidates present: ['Troponin', 'CK-MB']


## 8) Обучение модели

Сделаем два простых эксперимента:
1) Модель на **всех признаках**.
2) Модель **без подозрительных признаков** (`Troponin`, `CK-MB`).

Метрики:
- `F1` по классу (простая метрика качества классификации)
- `ROC-AUC` по вероятностям (часто полезна при дисбалансе)


In [7]:
def train_and_score(X: pd.DataFrame, y: pd.Series, cat_features):
    X_train, X_val, y_train, y_val = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )
    model = CatBoostClassifier(
        iterations=400,
        random_seed=42,
        verbose=100,
    )
    model.fit(X_train, y_train, cat_features=cat_features)
    val_pred = model.predict(X_val).astype(int)
    val_proba = model.predict_proba(X_val)[:, 1]
    return model, {
        "f1": float(f1_score(y_val, val_pred)),
        "roc_auc": float(roc_auc_score(y_val, val_proba)),
    }

y = train_p[target_col].astype(int)

# (1) all features
feature_cols_all = [c for c in train_p.columns if c not in [target_col, id_col]]
X_all = train_p[feature_cols_all]
cat_features = ["Gender"] if "Gender" in feature_cols_all else []

model_all, metrics_all = train_and_score(X_all, y, cat_features)
print("ALL FEATURES metrics:", metrics_all)

Learning rate set to 0.054616
0:	learn: 0.6843582	total: 60.6ms	remaining: 24.2s
100:	learn: 0.5851873	total: 301ms	remaining: 892ms
200:	learn: 0.5366636	total: 527ms	remaining: 522ms
300:	learn: 0.4865346	total: 754ms	remaining: 248ms
399:	learn: 0.4449544	total: 977ms	remaining: 0us
ALL FEATURES metrics: {'f1': 0.12411847672778561, 'roc_auc': 0.5681878672481215}


In [8]:
# (2) without leak candidates
drop_cols = [c for c in ["Troponin", "CK-MB"] if c in feature_cols_all]
feature_cols_noleak = [c for c in feature_cols_all if c not in drop_cols]
X_noleak = train_p[feature_cols_noleak]
cat_features_noleak = ["Gender"] if "Gender" in feature_cols_noleak else []

model_noleak, metrics_noleak = train_and_score(X_noleak, y, cat_features_noleak)
print("NO-LEAK FEATURES metrics:", metrics_noleak)
print("Dropped columns:", drop_cols)

Learning rate set to 0.054616
0:	learn: 0.6843334	total: 4.04ms	remaining: 1.61s
100:	learn: 0.5870709	total: 228ms	remaining: 674ms
200:	learn: 0.5350107	total: 447ms	remaining: 443ms
300:	learn: 0.4868494	total: 671ms	remaining: 221ms
399:	learn: 0.4448053	total: 889ms	remaining: 0us
NO-LEAK FEATURES metrics: {'f1': 0.12034383954154727, 'roc_auc': 0.5697102377588834}
Dropped columns: ['Troponin', 'CK-MB']


## 9) Выбор финальной модели

Если метрики на всех признаках сильно выше, чем без `Troponin/CK-MB`, это может быть сигналом,
что часть качества объясняется утечкой.

В проекте обычно честнее использовать модель **без подозрительных признаков**.


## 10) Предсказания для тестовой выборки

Формат сдачи: CSV с колонками `id` и `prediction`.

Важно: твой `test.py` читает ответ как `pd.read_csv(..., index_col=0)`.
Поэтому мы сохраняем CSV **с индексом**.

In [9]:
# Save predictions to project_root/outputs/student_predictions.csv
out_path = project_root / "outputs" / "student_predictions.csv"
out_path.parent.mkdir(parents=True, exist_ok=True)

# выбираем модель (без утечки, если возможно)
use_model = model_noleak
use_features = feature_cols_noleak

X_test = test_p[use_features]
test_pred = use_model.predict(X_test).astype(int)

pred_df = pd.DataFrame({"id": test_p[id_col].values, "prediction": test_pred})
pred_df.to_csv(out_path, index=True)

print("Saved:", out_path)
pred_df.head()

Saved: /Users/reckless/projects/heart_m1/outputs/student_predictions.csv


,id,prediction
0,7746,0
1,4202,0
2,6632,0
3,4639,0
4,4825,0


## 11) Итоги

Что сделано:
- первичный осмотр датасета (пропуски, типы, баланс классов)
- минимальная предобработка (`Unnamed: 0`, `Gender`)
- проверка на сильные корреляции
- обсуждение и простая проверка кандидатов на утечку (`Troponin`, `CK-MB`)
- обучение CatBoost и сравнение качества (all vs no-leak)
- генерация `student_predictions.csv`

Дальше (если есть время) можно улучшать качество:
- подбор гиперпараметров
- кросс-валидация
- более аккуратная работа с типами/категориями
- проверка важности признаков